# Waste Detection 2-Stage Pipeline (Final)

Notebook thuc thi toan bo quy trinh tren Kaggle.

### Kien truc:
1. **Stage 1:** YOLO26s phát hiện vùng rác (binary), `conf=0.40`
2. **Stage 2:** EfficientNet-B2 phân loại 6 lớp (5 rac + Background loc FP)

### Cau hinh:
| Thanh phan | Gia tri |
|---|---|
| Stage 1 model | YOLO26s (150 epochs) |
| Stage 1 conf | **0.40** |
| Stage 2 model | EfficientNet-B2 |
| Stage 2 Dropout | 0.5 |
| Mixup alpha | 0.3 |
| TTA | 5 augments |
| Macro F1 | Tinh tren 5 lop rac (khong tinh Background) |
| Model Stage 1 | models/final_best.pt |
| Model Stage 2 | models/final_best.pth |

**Yeu cau:** Import file lên Kaggle, bật internet và GPU T4 x2 để tối ưu tốc độ train.

In [ ]:
# ============================================================
# 1. Tai Ma nguon & Cai dat thu vien
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics timm

In [ ]:
# ============================================================
# 2. Nap Du lieu Ngoai lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Da tai: {os.path.basename(task['src'])}")
    else:
        print(f"Bo qua: {task['src']} (Khong tim thay tren Kaggle Dataset)")

In [ ]:
# ============================================================
# 3. Tien xu ly Du lieu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("--- 3.1 Don dep & Tao nhan YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("--- 3.2 Chuan bi du lieu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("Hoan tat chuan bi du lieu!")

In [ ]:
if not Path("models/final_best.pt").exists():
    !python src/train_stage1_detector.py   # gọn, nhất quán với cell 5
else:
    print("Da tim thay models/final_best.pt, bo qua.")

In [ ]:
# ============================================================
# 4b. Danh gia Stage 1 (YOLO Binary Detector)
# Kiem tra YOLO detect bao nhieu % rac truoc Stage 2
# ============================================================

!python src/evaluate_stage1.py \
  --detector models/final_best.pt \
  --data-dir data/processed_binary/images/test \
  --label-dir data/processed_binary/labels/test \
  --conf 0.40 \
  --iou 0.5 \
  --output results/stage1_eval

print("Danh gia Stage 1 hoan tat! Ket qua tai results/stage1_eval")

In [ ]:
# ============================================================
# 5. Huan luyen Stage 2 (EfficientNet-B2 - 6 Lop Classifier)
# Dropout=0.5 | WeightDecay=1e-4 | Mixup(alpha=0.3) | TTA
# Luu ra: models/final_best.pth
# ============================================================
!python src/train_stage2_classifier.py

In [ ]:
# ============================================================
# 6. Danh gia Toan Trinh (End-to-End 2-Stage Pipeline)
# conf=0.40 | Macro F1 tinh tren 5 lop rac (khong tinh Background)
# ============================================================

%cd /kaggle/working/waste-detection2-Stage

!python src/evaluate_2stage.py \
    --detector models/final_best.pt \
    --classifier models/final_best.pth \
    --data-dir data/processed/images/test \
    --label-dir data/processed/labels/test \
    --conf 0.4 \
    --output results/eval_final_v2


In [ ]:
# ============================================================
# 7. Zip ket qua de tai ve
# ============================================================
import shutil, os

shutil.make_archive("/kaggle/working/final_results", "zip",
                    "/kaggle/working/waste-detection2-Stage/results")
print("Da nen ket qua: /kaggle/working/final_results.zip")

if os.path.exists("/kaggle/working/waste-detection2-Stage/models"):
    shutil.make_archive("/kaggle/working/final_models", "zip",
                        "/kaggle/working/waste-detection2-Stage/models")
    print("Da nen model: /kaggle/working/final_models.zip")